# MedNorm E5 XLM-R MRC-NER Training

Status: IMPLEMENTED_UNTRAINED. Query conversion and offset pairing run before any model weight acquisition.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import random

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts" / "phase2_e5_xlmr_mrc_ner"
SMOKE_OUTPUT_DIR = ARTIFACT_ROOT / "smoke"
FULL_OUTPUT_DIR = ARTIFACT_ROOT / "full_training"
MODEL_CACHE_DIR = DRIVE_ROOT / "model_cache" / "huggingface"
RUN_FULL_TRAINING = False
CONFIRM_FULL = ""
RESUME_FROM_SMOKE_CHECKPOINT = False
SEED = 20260727
PINNED_MODEL_REVISION = "UNPINNED_UNTRAINED_REQUIRES_OPERATOR_PIN"
OUTPUT_DIR = FULL_OUTPUT_DIR if RUN_FULL_TRAINING else SMOKE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)
assert not (RUN_FULL_TRAINING and RESUME_FROM_SMOKE_CHECKPOINT)
if RUN_FULL_TRAINING:
    assert CONFIRM_FULL == "I_AUTHORIZE_E5_FULL_TRAINING"
    assert PINNED_MODEL_REVISION != "UNPINNED_UNTRAINED_REQUIRES_OPERATOR_PIN"


In [ ]:
EXPECTED_CORPUS_HASHES = {
    "public_ner_train.jsonl": "892dc22d7e051e05f9c96d90f42dfde7f38083a74bba6fe65b5c1d9dd05e2a4a",
    "public_ner_validation.jsonl": "ed7cdd2d49799cef0a868b6c75a3df4ca1e93ed03223337a7d31afe40f68f103",
}
CORPUS_DIR = DRIVE_ROOT / "data" / "processed"

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def validate_corpus_hashes(corpus_dir: Path) -> dict[str, str]:
    observed = {}
    for name, expected in EXPECTED_CORPUS_HASHES.items():
        path = corpus_dir / name
        assert path.is_file(), f"missing governed corpus file: {path}"
        digest = sha256_file(path)
        assert digest == expected, f"hash mismatch for {name}"
        observed[name] = digest
    return observed

corpus_hashes = validate_corpus_hashes(CORPUS_DIR)


In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR / "src"))
from mednorm_vi.mention_factory.mrc import build_mrc_examples, pair_start_end
from mednorm_vi.mention_factory.w2ner import EntitySpan

text = "Bệnh nhân suy tim và ho khan"
gold = (EntitySpan(text.index("suy tim"), text.index("suy tim") + len("suy tim"), "DIAGNOSIS", "suy tim"),)
examples = build_mrc_examples("smoke-e5", text, gold)
diagnosis = next(example for example in examples if example.entity_type == "DIAGNOSIS")
start_scores = [float(label) for label in diagnosis.start_labels]
end_scores = [float(label) for label in diagnosis.end_labels]
decoded = pair_start_end(diagnosis, start_scores, end_scores, threshold=1.0)
assert [(s.start, s.end, s.entity_type) for s in decoded] == [(gold[0].start, gold[0].end, "DIAGNOSIS")]
assert all(not diagnosis.query_mask[i] for i, label in enumerate(diagnosis.start_labels) if label)
preflight_report = {"query_count": len(examples), "decoded_spans": len(decoded), "corpus_hashes": corpus_hashes}
(OUTPUT_DIR / "preflight_report.json").write_text(json.dumps(preflight_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


In [ ]:
from mednorm_vi.training.manifests import RunManifest

manifest = RunManifest(
    stage="smoke" if not RUN_FULL_TRAINING else "training",
    expert="E5_xlmr_mrc_ner",
    config_sha256=hashlib.sha256(b"xlmr-mrc-ner-v1").hexdigest(),
    data_sha256=hashlib.sha256(json.dumps(corpus_hashes, sort_keys=True).encode()).hexdigest(),
    corpus_sha256=corpus_hashes["public_ner_train.jsonl"],
    model_revision=PINNED_MODEL_REVISION,
    seed=SEED,
    git_commit="COLAB_RESOLVES_BEFORE_FULL_TRAINING",
    checkpoint_sha256="UNAVAILABLE_UNTRAINED" if not RUN_FULL_TRAINING else "WRITTEN_AFTER_SAVE_RELOAD_VALIDATION",
    parameter_count=560000000,
    train_split_id="public_ner_train_governed_v1",
    validation_split_id="public_ner_validation_governed_v1",
    internal_test_accessed=False,
)
manifest.validate()
manifest.write_json(OUTPUT_DIR / "run_manifest.json")

def validate_checkpoint_after_save_reload(path: Path, expected_sha256: str) -> None:
    assert path.is_file()
    assert sha256_file(path) == expected_sha256

if RUN_FULL_TRAINING:
    checkpoint_path = OUTPUT_DIR / "checkpoint" / "best.pt"
    validate_checkpoint_after_save_reload(checkpoint_path, manifest.checkpoint_sha256)
